# Cobertura Wi-Fi — Predios M e I

**CC0048 — Redes Sem Fio** · UFCA

Este notebook **orquestra**; ele nao implementa. Toda a logica esta em `src/`, e toda
a configuracao em `config/predios.py`. O notebook existe para executar na ordem certa
e exibir os resultados.

## Arquitetura

| Modulo | Responsabilidade |
|---|---|
| `config/predios.py` | Unica fonte de valores por predio. Nenhum nome de predio aparece em `src/` |
| `src/esquema.py` | Carga e validacao de `dados/leituras.csv` |
| `src/bssid.py` | Normalizacao, agrupamento em AP fisico, deteccao de transcricao, dominancia |
| `src/geometria.py` | Distancia 3D, AP mais proximo, resolucao de coordenadas |
| `src/pathloss.py` | Os 4 cenarios de ajuste, IC 95%, atenuacao por obstaculo |
| `src/laje.py` | Perda entre pavimentos, por pares do mesmo AP fisico |
| `src/canais.py` | Descasamento, reuso e qualidade relativa de canal |
| `src/heatmap.py` | Mapas por pavimento, IDW com raio maximo, GeoTIFF |
| `src/qualidade.py` | Relatorio de qualidade — roda **primeiro** |
| `src/limitacoes.py` | `saidas/limitacoes.md`, montado dos dados reais |

## Para acrescentar um terceiro predio

Uma entrada nova em `PREDIOS` e as linhas no CSV. **Nada em `src/` muda.**

In [1]:
# =============================================================================
# 1. CARGA, VALIDACAO, BSSID E GEOMETRIA
# =============================================================================
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

from config.predios import PARAMS, PREDIOS
from src import pipeline

df = pipeline.preparar()

print("Leituras carregadas :", len(df))
print("Predios             :", ", ".join(sorted(df.predio.unique())))
print("Bandas              :", ", ".join(sorted(df.banda.unique())))
print()
display(df.groupby(["predio", "pavimento", "banda"]).size().rename("leituras").reset_index())

Leituras carregadas : 56
Predios             : I, M
Bandas              : 2.4, 5



,predio,pavimento,banda,leituras
0,I,-1,2.4,5
1,I,-1,5,4
2,I,0,2.4,5
3,I,0,5,3
4,I,1,2.4,5
5,I,1,5,4
6,M,-1,2.4,6
7,M,-1,5,4
8,M,0,2.4,6
9,M,0,5,4


---
## Dominancia de AP e BSSIDs suspeitos

Os BSSIDs vem da lista de varredura do aplicativo: identificam o **AP dominante** no
ponto, nao o AP ao qual o cliente estava associado. Toda a saida usa esse vocabulario.

A classificacao de dominancia e o **unico** criterio de exclusao permitido na
regressao de path loss local. Nao existe exclusao por ID de ponto no codigo.

In [2]:
from src.bssid import detectar_bssids_suspeitos, grupos_candidatos

print("Dominancia do AP por predio:")
display(df.groupby(["predio", "dominancia"]).size().rename("leituras").reset_index())

suspeitos = detectar_bssids_suspeitos(df, PREDIOS)
print("\nPares de BSSID sinalizados, com a hipotese mais provavel:")
display(suspeitos[["predio", "bssid_a", "leituras_a", "bssid_b", "leituras_b",
                   "hamming_nibbles", "hipotese", "bssid_anomalo"]])

for predio in sorted(df.predio.unique()):
    if not (PREDIOS.get(predio) or {}).get("bssid_para_ap"):
        print(f"\nGrupos candidatos de AP fisico — predio {predio} "
              f"(aguardando conferencia manual):")
        display(grupos_candidatos(df, predio, PREDIOS))

Dominancia do AP por predio:


,predio,dominancia,leituras
0,I,sem_dado,26
1,M,ap_nao_mapeado,1
2,M,local,2
3,M,outro_pavimento,1
4,M,sem_dado,25
5,M,suspeito,1



Pares de BSSID sinalizados, com a hipotese mais provavel:


,predio,bssid_a,leituras_a,bssid_b,leituras_b,hamming_nibbles,hipotese,bssid_anomalo
0,I,00:e6:3a:5e:4e:a0,[32],00:e6:3a:9e:4e:a0,"[46, 47]",1,offset de radio 2.4/5 GHz no 4o octeto,NaN
1,I,3c:46:a1:66:33:30,"[37, 44]",3c:46:a1:66:33:40,"[34, 36, 38, 39, 40, 41, 42]",1,"mesmo AP fisico (prefixo de 5 octetos, radio/SSID distinto)",NaN
2,I,3c:46:a1:66:33:30,"[37, 44]",3c:46:a1:a6:33:40,"[50, 51, 52, 53, 54]",2,offset de radio 2.4/5 GHz no 4o octeto,3c:46:a1:a6:33:40
3,I,3c:46:a1:66:33:40,"[34, 36, 38, 39, 40, 41, 42]",3c:46:a1:a6:33:40,"[50, 51, 52, 53, 54]",1,offset de radio 2.4/5 GHz no 4o octeto,3c:46:a1:a6:33:40
4,I,c8:a6:08:43:20:a0,[35],c8:a6:08:83:20:a0,"[48, 49]",1,offset de radio 2.4/5 GHz no 4o octeto,NaN
5,M,e0:10:7f:3d:ea:78,[7],e0:10:7f:7d:ea:79,[14],2,possivel erro de transcricao,e0:10:7f:7d:ea:79



Grupos candidatos de AP fisico — predio I (aguardando conferencia manual):


,grupo,bssids,oui,leituras,pavimentos,bandas,locais
0,00:e6:3a:5e:4e:a0,00:e6:3a:5e:4e:a0,00:e6:3a,[32],[1],[2.4],Abaixo do AP2
1,00:e6:3a:8a:81:70,00:e6:3a:8a:81:70,00:e6:3a,[56],[-1],[5],Canto da Biblioteca
2,00:e6:3a:9e:4e:a0,00:e6:3a:9e:4e:a0,00:e6:3a,"[46, 47]",[1],[5],Abaixo do AP2; Fundo do corredor
3,3c:46:a1:66:33:30,3c:46:a1:66:33:30,3c:46:a1,"[37, 44]","[-1, 0]",[2.4],Abaixo do AP2; Canto da Biblioteca
4,3c:46:a1:66:33:40,3c:46:a1:66:33:40,3c:46:a1,"[34, 36, 38, 39, 40, 41, 42]","[-1, 0, 1]",[2.4],Abaixo do AP1; Entrada; Entrada/Bebedouro; Escada; Fundo do Terreo...
5,3c:46:a1:a6:33:40,3c:46:a1:a6:33:40,3c:46:a1,"[50, 51, 52, 53, 54]","[-1, 0]",[5],Abaixo do AP1; Abaixo do AP2; Entrada/Bebedouro; Fundo do Terreo
6,70:47:77:74:84:10,70:47:77:74:84:10,70:47:77,[31],[1],[2.4],Fundo do corredor
7,70:47:77:75:2a:a0,70:47:77:75:2a:a0,70:47:77,[33],[1],[2.4],Abaixo do AP1
8,70:47:77:b4:55:60,70:47:77:b4:55:60,70:47:77,[55],[-1],[5],Abaixo do AP2
9,c8:a6:08:43:20:a0,c8:a6:08:43:20:a0,c8:a6:08,[35],[1],[2.4],Escada


---
## 2. Relatorio de qualidade — roda ANTES das analises

A saida mais importante do projeto neste momento nao e um alpha: e a lista objetiva do
que precisa ser recoletado em campo. O arquivo completo fica em
`saidas/relatorio_qualidade.md`.

In [3]:
# Executa o pipeline inteiro. O relatorio de qualidade e gravado dentro dele.
S = pipeline.rodar()
df = S["df"]

from IPython.display import Markdown
Markdown(S["relatorio_qualidade"])

# Relatorio de qualidade dos dados

Gerado automaticamente em 18/08/2026 17:43 a partir de `dados/leituras.csv`.

Este relatorio roda **antes** de qualquer analise. Nenhuma leitura e descartada pelo pipeline sem aparecer em alguma secao abaixo.


## 1. Contagem de leituras

| predio   | pavimento   |   banda |   leituras |
|:---------|:------------|--------:|-----------:|
| I        | Subsolo     |     2.4 |          5 |
| I        | Subsolo     |     5   |          4 |
| I        | Terreo      |     2.4 |          5 |
| I        | Terreo      |     5   |          3 |
| I        | 1o andar    |     2.4 |          5 |
| I        | 1o andar    |     5   |          4 |
| M        | Subsolo     |     2.4 |          6 |
| M        | Subsolo     |     5   |          4 |
| M        | Terreo      |     2.4 |          6 |
| M        | Terreo      |     5   |          4 |
| M        | 1o andar    |     2.4 |          5 |
| M        | 1o andar    |     5   |          5 |

**Total: 56 leituras.**


## 2. Campos ausentes por coluna

| coluna                 |   ausentes |   total |   pct | critica   |
|:-----------------------|-----------:|--------:|------:|:----------|
| altura_medicao_m       |         56 |      56 | 100   | SIM       |
| x_m                    |         32 |      56 |  57.1 | SIM       |
| y_m                    |         32 |      56 |  57.1 | SIM       |
| bssid_bruto            |         25 |      56 |  44.6 | SIM       |
| dist_ao_ap_dominante_m |         54 |      56 |  96.4 |           |
| dist_ap_conectado_m    |         53 |      56 |  94.6 |           |
| ap_dominante           |         53 |      56 |  94.6 |           |
| delta_pavimento        |         53 |      56 |  94.6 |           |
| ap_local               |         32 |      56 |  57.1 |           |
| dist_calc_2d_m         |         32 |      56 |  57.1 |           |
| dist_calc_3d_m         |         32 |      56 |  57.1 |           |
| divergencia_dist_m     |         32 |      56 |  57.1 |           |
| bssid                  |         25 |      56 |  44.6 |           |
| grupo_ap               |         25 |      56 |  44.6 |           |
| rssi_dbm               |          1 |      56 |   1.8 |           |
| canal_usado            |          1 |      56 |   1.8 |           |

> As colunas marcadas como criticas sustentam analises inteiras: `bssid_bruto` decide a dominancia de AP (cenarios B/D e perda de laje) e `x_m`/`y_m` decidem toda a geometria (cenarios C/D e mapas).


## 3. Origem da distancia — risco de circularidade

| predio   | dist_origem   |   leituras |
|:---------|:--------------|-----------:|
| I        | planta        |         26 |
| M        | planta        |         30 |

> **Nenhuma leitura com `dist_origem = 'estimada_app'`.** Nao ha risco de circularidade: a regressao nao recupera o modelo interno do aplicativo. As distancias declaradas como `planta` foram lidas do projeto arquitetonico, e sao independentes do RSSI medido.


## 4. BSSIDs suspeitos de erro de transcricao

| predio   | bssid_a           | leituras_a                   | bssid_b           | leituras_b                   |   hamming_nibbles | octetos_divergentes   | hipotese                                                    | bssid_anomalo     |
|:---------|:------------------|:-----------------------------|:------------------|:-----------------------------|------------------:|:----------------------|:------------------------------------------------------------|:------------------|
| I        | 00:e6:3a:5e:4e:a0 | [32]                         | 00:e6:3a:9e:4e:a0 | [46, 47]                     |                 1 | [4]                   | offset de radio 2.4/5 GHz no 4o octeto                      | nan               |
| I        | 3c:46:a1:66:33:30 | [37, 44]                     | 3c:46:a1:66:33:40 | [34, 36, 38, 39, 40, 41, 42] |                 1 | [6]                   | mesmo AP fisico (prefixo de 5 octetos, radio/SSID distinto) | nan               |
| I        | 3c:46:a1:66:33:30 | [37, 44]                     | 3c:46:a1:a6:33:40 | [50, 51, 52, 53, 54]         |                 2 | [4, 6]                | offset de radio 2.4/5 GHz no 4o octeto                      | 3c:46:a1:a6:33:40 |
| I        | 3c:46:a1:66:33:40 | [34, 36, 38, 39, 40, 41, 42] | 3c:46:a1:a6:33:40 | [50, 51, 52, 53, 54]         |                 1 | [4]                   | offset de radio 2.4/5 GHz no 4o octeto                      | 3c:46:a1:a6:33:40 |
| I        | c8:a6:08:43:20:a0 | [35]                         | c8:a6:08:83:20:a0 | [48, 49]                     |                 1 | [4]                   | offset de radio 2.4/5 GHz no 4o octeto                      | nan               |
| M        | e0:10:7f:3d:ea:78 | [7]                          | e0:10:7f:7d:ea:79 | [14]                         |                 2 | [4, 6]                | possivel erro de transcricao                                | e0:10:7f:7d:ea:79 |

> Sinalizado todo par com distancia de Hamming (em nibbles) <= 2 que a regra de agrupamento do predio separa em APs distintos. **Apenas os 1 par(es) classificados como erro de transcricao marcam a leitura como suspeita**; os demais indicam que a regra de agrupamento daquele predio precisa de revisao, nao que o dado esteja errado.


### Grupos candidatos de AP fisico — predio I

O mapeamento `bssid_para_ap` deste predio esta vazio: a regra de agrupamento confirmada para outro predio **nao foi assumida aqui**. Confira os grupos abaixo e preencha `config/predios.py` manualmente.

| grupo             | bssids            | oui      | leituras                     | pavimentos   | bandas   | locais                                                                               |
|:------------------|:------------------|:---------|:-----------------------------|:-------------|:---------|:-------------------------------------------------------------------------------------|
| 00:e6:3a:5e:4e:a0 | 00:e6:3a:5e:4e:a0 | 00:e6:3a | [32]                         | [1]          | ['2.4']  | Abaixo do AP2                                                                        |
| 00:e6:3a:8a:81:70 | 00:e6:3a:8a:81:70 | 00:e6:3a | [56]                         | [-1]         | ['5']    | Canto da Biblioteca                                                                  |
| 00:e6:3a:9e:4e:a0 | 00:e6:3a:9e:4e:a0 | 00:e6:3a | [46, 47]                     | [1]          | ['5']    | Abaixo do AP2; Fundo do corredor                                                     |
| 3c:46:a1:66:33:30 | 3c:46:a1:66:33:30 | 3c:46:a1 | [37, 44]                     | [-1, 0]      | ['2.4']  | Abaixo do AP2; Canto da Biblioteca                                                   |
| 3c:46:a1:66:33:40 | 3c:46:a1:66:33:40 | 3c:46:a1 | [34, 36, 38, 39, 40, 41, 42] | [-1, 0, 1]   | ['2.4']  | Abaixo do AP1; Entrada; Entrada/Bebedouro; Escada; Fundo do Terreo; Janela/Bebedouro |
| 3c:46:a1:a6:33:40 | 3c:46:a1:a6:33:40 | 3c:46:a1 | [50, 51, 52, 53, 54]         | [-1, 0]      | ['5']    | Abaixo do AP1; Abaixo do AP2; Entrada/Bebedouro; Fundo do Terreo                     |
| 70:47:77:74:84:10 | 70:47:77:74:84:10 | 70:47:77 | [31]                         | [1]          | ['2.4']  | Fundo do corredor                                                                    |
| 70:47:77:75:2a:a0 | 70:47:77:75:2a:a0 | 70:47:77 | [33]                         | [1]          | ['2.4']  | Abaixo do AP1                                                                        |
| 70:47:77:b4:55:60 | 70:47:77:b4:55:60 | 70:47:77 | [55]                         | [-1]         | ['5']    | Abaixo do AP2                                                                        |
| c8:a6:08:43:20:a0 | c8:a6:08:43:20:a0 | c8:a6:08 | [35]                         | [1]          | ['2.4']  | Escada                                                                               |
| c8:a6:08:44:3b:70 | c8:a6:08:44:3b:70 | c8:a6:08 | [43, 45]                     | [-1]         | ['2.4']  | Abaixo do AP2; Escada                                                                |
| c8:a6:08:83:20:a0 | c8:a6:08:83:20:a0 | c8:a6:08 | [48, 49]                     | [1]          | ['5']    | Abaixo do AP1; Janela/Bebedouro                                                      |

## 5. Leituras sem RSSI

| ponto_id   | predio   |   pavimento |   banda | local   | obs_campo                                                                                                                                       |
|:-----------|:---------|------------:|--------:|:--------|:------------------------------------------------------------------------------------------------------------------------------------------------|
| M-22       | M        |           1 |       5 | Escada  | Sem conexao em 5 GHz na escada: nao foi possivel identificar associacao com nenhum AP, provavelmente por distancia ou interferencia. Zona cega. |

## 6. Leituras com distancia de campo <= 0

| ponto_id   | predio   |   pavimento |   banda |   dist_campo_m | local         | obs_campo                                                                                                                                                                                                                                                                                               |
|:-----------|:---------|------------:|--------:|---------------:|:--------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| I-32       | I        |           1 |     2.4 |              0 | Abaixo do AP2 | Distancia anotada em campo como 0 m (medicao diretamente sob o AP).                                                                                                                                                                                                                                     |
| I-43       | I        |          -1 |     2.4 |              0 | Abaixo do AP2 | Distancia anotada em campo como 0 m (medicao diretamente sob o AP). Reclassificado como SEM obstaculo em 18/08/2026: a medicao foi feita diretamente sob o AP, sem nada interposto no caminho do enlace; as prateleiras de metal estao no entorno do ponto. Anotacao original em obstaculos_registrado. |
| I-47       | I        |           1 |     5   |              0 | Abaixo do AP2 | Distancia anotada em campo como 0 m (medicao diretamente sob o AP).                                                                                                                                                                                                                                     |
| I-55       | I        |          -1 |     5   |              0 | Abaixo do AP2 | Distancia anotada em campo como 0 m (medicao diretamente sob o AP). Reclassificado como SEM obstaculo em 18/08/2026: mesmo criterio do ponto 43. Anotacao original em obstaculos_registrado.                                                                                                            |

> Anotadas em campo como 0 m (medicao diretamente sob o AP). `log10(0)` e indefinido, entao essas leituras **saem de toda regressao** — o pipeline nao as normaliza para d0 em silencio. Recoletar com a distancia horizontal real ao AP resolveria.


## 7. Divergencia entre distancia de campo e geometria 3D

Limite de reporte: **3 m**.

_(nenhuma divergencia acima do limite — ou geometria indisponivel)_

### 7b. Leituras com DUAS distancias anotadas em campo

| ponto_id   | predio   |   pavimento |   banda |   dist_campo_m |   dist_ap_conectado_m |   rssi_dbm | local   |
|:-----------|:---------|------------:|--------:|---------------:|----------------------:|-----------:|:--------|
| M-15       | M        |          -1 |     2.4 |              6 |                    15 |        -60 | Entrada |
| M-16       | M        |          -1 |     2.4 |              8 |                    18 |        -68 | Escada  |
| M-22       | M        |           1 |     5   |              8 |                    18 |        nan | Escada  |

> A campanha anotou distancia real **e** distancia ao AP dominante. `dist_campo_m` recebe a real (lida da planta); a outra fica em `dist_ap_conectado_m`. A escolha muda o alpha de forma material e esta reportada na tabela comparativa de cenarios.


## 8. Executabilidade das analises

Esta e a lista objetiva do que precisa ser recoletado em campo.

| analise                                        | situacao                | motivo                                                                                                                                 | o_que_falta_coletar                                                                                                           |
|:-----------------------------------------------|:------------------------|:---------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------------------|
| 1. Path loss — cenario A (distancia de campo)  | executavel com ressalva | estimado em 2 de 4 combinacoes; sem estimativa em: I 5 GHz, M 5 GHz                                                                    | leituras sem obstaculo em distancias intermediarias (6, 10 e 15 m), sobretudo em 5 GHz, onde ha apenas 2 distancias distintas |
| 2. Path loss — cenarios B/D (dominancia de AP) | bloqueada               | estimado em 0 de 8 combinacoes; cobertura de bssid_bruto parcial (M: 5/30); mapa de AP vazio em: I                                     | BSSID em TODAS as leituras (faltam 25 de 56) e confirmacao manual dos grupos de AP dos predios sem mapa                       |
| 3. Path loss — cenarios C/D (geometria 3D)     | bloqueada               | estimado em 0 de 8 combinacoes; x/y so sao conhecidos onde o local e 'Abaixo do APn' (I: 12/26, M: 12/30)                              | x_m e y_m lidos em planta para TODOS os pontos (duas distancias perpendiculares por ponto; precisao de +-0,5 m basta)         |
| 4. Perda de laje                               | bloqueada               | minimo de 3 pares utilizaveis por combinacao; obtido — I 2.4 GHz: 15, I 5 GHz: 6, M 2.4 GHz: 1, M 5 GHz: 0                             | BSSID em todas as leituras, para achar o mesmo AP fisico visto de pavimentos diferentes; e x/y, para descontar a distancia    |
| 5. Atenuacao por obstaculo                     | executavel com ressalva | calculada em 2 de 4 combinacoes; 3 ponto(s) com L <= 0, sinalizados e nao recategorizados                                              | alpha valido em 5 GHz — hoje sem alavanca em distancia — para estender o calculo aquela banda                                 |
| 6. Canais (descasamento, reuso, qualidade)     | executavel              | descasamento calculado em 12 grupos predio x banda x pavimento; reuso identificavel apenas onde ha BSSID                               | definicao explicita da semantica de pct_melhor_canal; BSSID completo para fechar o mapa de reuso de canal                     |
| 7. Mapas de calor por pavimento                | bloqueada               | 12 mapa(s) gerado(s), 0 com superficie interpolada; os demais saem como scatter porque ha menos de 4 posicoes distintas com coordenada | x/y de todos os pontos e as imagens de planta por pavimento                                                                   |

---


## Avisos de semantica

- SEMANTICA PENDENTE: 'pct_melhor_canal' tem interpretacao ambigua (qualidade do melhor canal vs. ocupacao do melhor canal). O pipeline a trata APENAS como metrica relativa comparativa, nunca como grandeza fisica. Defina explicitamente no relatorio antes de citar qualquer valor absoluto.

- Os BSSIDs identificam o **AP dominante** na varredura, nao o AP ao qual o cliente estava associado. Nenhuma conclusao sobre associacao de cliente pode ser tirada deste dado.


---
## 3. Path loss — os 4 cenarios

| # | Distancia | Filtro |
|---|---|---|
| A | `dist_campo_m` | todos exceto `dist_origem == "estimada_app"` |
| B | `dist_campo_m` | apenas `dominancia == "local"` |
| C | `dist_calc_3d_m` | todos exceto `estimada_app` |
| D | `dist_calc_3d_m` | apenas `dominancia == "local"` |

Alem deles, duas linhas de apoio: o **cenario historico** (distancia como estava
gravada no CSV antigo), que serve a verificacao de regressao, e o **cenario B+**
(local ou sem BSSID), que existe porque a cobertura de BSSID e parcial.

`alpha` sai com **2 algarismos significativos**. Abaixo de
`PARAMS["min_pontos_ajuste"]` nao sai numero, sai o motivo.

In [4]:
cols = ["predio", "banda", "cenario", "alpha", "ic95_inf", "ic95_sup",
        "r2", "n", "rmse_db", "status"]
print("CENARIOS A-D")
display(S["cenarios"][cols])

print("\nMotivos, onde nao houve estimativa:")
for r in S["cenarios"][S["cenarios"].status != "estimado"].itertuples():
    print(f"  {r.predio} {r.banda} GHz · cenario {r.cenario} (n={r.n}): {r.motivo}")

print("\nCENARIO HISTORICO (verificacao) e DIAGNOSTICO B+")
display(pd.concat([S["cenarios_historico"], S["cenarios_diagnostico"]])[cols])

CENARIOS A-D


,predio,banda,cenario,alpha,ic95_inf,ic95_sup,r2,n,rmse_db,status
0,I,2.4,A,1.90,1.07,2.81,0.8322,8,2.96,estimado
1,I,2.4,B,NaN,NaN,NaN,NaN,0,NaN,nao estimavel
2,I,2.4,C,NaN,NaN,NaN,NaN,4,NaN,nao estimavel
3,I,2.4,D,NaN,NaN,NaN,NaN,0,NaN,nao estimavel
4,I,5,A,0.47,-2.10,3.04,0.0319,8,7.25,inconsistente
5,I,5,B,NaN,NaN,NaN,NaN,0,NaN,nao estimavel
6,I,5,C,NaN,NaN,NaN,NaN,4,NaN,nao estimavel
7,I,5,D,NaN,NaN,NaN,NaN,0,NaN,nao estimavel
8,M,2.4,A,3.30,1.91,4.62,0.8232,9,4.35,estimado
9,M,2.4,B,NaN,NaN,NaN,NaN,2,NaN,nao estimavel



Motivos, onde nao houve estimativa:
  I 2.4 GHz · cenario B (n=0): nenhuma leitura no conjunto
  I 2.4 GHz · cenario C (n=4): amostra insuficiente: n = 4, minimo 5
  I 2.4 GHz · cenario D (n=0): nenhuma leitura no conjunto
  I 5 GHz · cenario A (n=8): ajuste nao confiavel: R2 = 0.03, abaixo do minimo 0.30 — a distancia explica apenas 3% da variacao do RSSI
  I 5 GHz · cenario B (n=0): nenhuma leitura no conjunto
  I 5 GHz · cenario C (n=4): amostra insuficiente: n = 4, minimo 5
  I 5 GHz · cenario D (n=0): nenhuma leitura no conjunto
  M 2.4 GHz · cenario B (n=2): amostra insuficiente: n = 2, minimo 5
  M 2.4 GHz · cenario C (n=4): amostra insuficiente: n = 4, minimo 5
  M 2.4 GHz · cenario D (n=2): amostra insuficiente: n = 2, minimo 5
  M 5 GHz · cenario A (n=9): amostra concentrada: apenas 2 distancia(s) distinta(s), minimo 3 — sem alavanca em distancia, alpha descreve a dispersao da amostra, nao a perda de percurso
  M 5 GHz · cenario B (n=0): nenhuma leitura no conjunto
  M 5 GHz

,predio,banda,cenario,alpha,ic95_inf,ic95_sup,r2,n,rmse_db,status
0,I,2.4,H,1.90,1.07,2.81,0.8322,8,2.96,estimado
1,I,5,H,0.47,-2.10,3.04,0.0319,8,7.25,inconsistente
2,M,2.4,H,2.60,1.87,3.37,0.9062,9,3.17,estimado
3,M,5,H,NaN,NaN,NaN,NaN,9,NaN,nao estimavel
0,I,2.4,B+,1.90,1.07,2.81,0.8322,8,2.96,estimado
1,I,5,B+,0.47,-2.10,3.04,0.0319,8,7.25,inconsistente
2,M,2.4,B+,3.30,1.91,4.62,0.8232,9,4.35,estimado
3,M,5,B+,NaN,NaN,NaN,NaN,9,NaN,nao estimavel


In [5]:
# Pontos efetivamente usados em cada cenario — nenhuma exclusao fica implicita.
for r in S["cenarios"].itertuples():
    if r.n:
        print(f"{r.predio} {r.banda} GHz · {r.cenario} (n={r.n}): {r.pontos}")

I 2.4 GHz · A (n=8): I-31, I-33, I-34, I-36, I-37, I-38, I-41, I-42
I 2.4 GHz · C (n=4): I-33, I-37, I-38, I-42
I 5 GHz · A (n=8): I-46, I-48, I-49, I-50, I-51, I-52, I-53, I-54
I 5 GHz · C (n=4): I-48, I-51, I-52, I-54
M 2.4 GHz · A (n=9): M-02, M-03, M-04, M-07, M-08, M-09, M-11, M-15, M-17
M 2.4 GHz · B (n=2): M-07, M-08
M 2.4 GHz · C (n=4): M-02, M-03, M-07, M-08
M 2.4 GHz · D (n=2): M-07, M-08
M 5 GHz · A (n=9): M-19, M-20, M-21, M-24, M-25, M-26, M-28, M-29, M-30
M 5 GHz · C (n=6): M-19, M-20, M-24, M-25, M-28, M-29


---
## 4. Perda de laje

Pares de leituras do **mesmo AP fisico** em pavimentos diferentes. Com menos de
`PARAMS["min_pares_laje"]` pares o resultado e "nao estimavel" — nunca o valor de um
unico par. Mediana negativa ou proxima de zero e reportada com alerta, sem ser forcada.

Os pares individuais sao sempre impressos, para inspecao manual.

In [6]:
display(S["laje"])
print("\nPares individuais:")
display(S["laje_pares"])

,predio,banda,cenario_alpha,alpha_usado,L_laje_mediana_db,ic95_inf,ic95_sup,n_pares,status,motivo
0,I,2.4,A,1.9,-1.2,-1.9,-0.2,15,inconsistente,estimativa inconsistente — a referencia de RSSI a 1 m sob o AP e i...
1,I,5,A,NaN,NaN,NaN,NaN,6,nao estimavel,sem alpha valido para a combinacao (ajuste nao confiavel: R2 = 0.0...
2,M,2.4,A,3.3,NaN,NaN,NaN,1,nao estimavel,"apenas 1 par(es) utilizavel(is), minimo 3 — um unico par nao suste..."
3,M,5,A,NaN,NaN,NaN,NaN,0,nao estimavel,nenhum par de leituras do mesmo AP fisico em pavimentos diferentes



Pares individuais:


,predio,banda,grupo_ap,ponto_a,pav_a,rssi_a,dist_a_m,origem_dist_a,local_a,ponto_b,pav_b,rssi_b,dist_b_m,origem_dist_b,local_b,delta_pavimentos,delta_rssi_db,L_laje_db,utilizavel,alpha_usado
0,I,2.4,3c:46:a1:66:33:30,I-37,0,-46.0,2.0,campo,Abaixo do AP2,I-44,-1,-56.0,6.0,campo,Canto da Biblioteca,1,10.0,0.8,True,1.938
1,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-36,0,-52.0,4.0,campo,Fundo do Terreo,1,6.0,-0.2,True,1.938
2,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-38,0,-36.0,1.0,campo,Abaixo do AP1,1,22.0,-4.5,True,1.938
3,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-39,0,-55.0,6.0,campo,Escada,1,3.0,-0.6,True,1.938
4,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-40,0,-59.0,9.0,campo,Entrada,1,1.0,0.0,True,1.938
5,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-41,-1,-45.0,2.0,campo,Entrada/Bebedouro,2,13.0,-1.3,True,1.938
6,I,2.4,3c:46:a1:66:33:40,I-34,1,-58.0,8.0,campo,Janela/Bebedouro,I-42,-1,-38.0,1.0,campo,Abaixo do AP1,2,20.0,-2.5,True,1.938
7,I,2.4,3c:46:a1:66:33:40,I-36,0,-52.0,4.0,campo,Fundo do Terreo,I-41,-1,-45.0,2.0,campo,Entrada/Bebedouro,1,7.0,-1.2,True,1.938
8,I,2.4,3c:46:a1:66:33:40,I-36,0,-52.0,4.0,campo,Fundo do Terreo,I-42,-1,-38.0,1.0,campo,Abaixo do AP1,1,14.0,-2.3,True,1.938
9,I,2.4,3c:46:a1:66:33:40,I-38,0,-36.0,1.0,campo,Abaixo do AP1,I-41,-1,-45.0,2.0,campo,Entrada/Bebedouro,1,9.0,3.2,True,1.938


---
## 5. Canais

Hipotese sob teste: o descasamento em 2,4 GHz e sistematico nos **dois** predios, o
que apontaria para ausencia de gerenciamento adaptativo de canal em nivel de campus,
e nao para um problema local.

In [7]:
print("Descasamento por predio x banda:")
display(S["mismatch_banda"])
print("\nPor pavimento:")
display(S["mismatch"])

h = S["hipotese_canal"]
print("\nHIPOTESE (2,4 GHz):", h.extra.get("leitura", h.motivo))

print("\nReuso de canal entre APs do mesmo predio:")
display(S["reuso"])

print("\nDistribuicao de pct_melhor_canal:")
display(S["pct"])
print("\n" + S["aviso_pct"])

Descasamento por predio x banda:


,predio,banda,leituras,descasadas,taxa_pct
0,I,2.4,15,14,93.3
1,I,5,11,11,100.0
2,M,2.4,17,13,76.5
3,M,5,12,12,100.0



Por pavimento:


,predio,banda,pavimento,leituras,descasadas,taxa_pct,pavimento_rotulo
0,I,2.4,-1,5,4,80.0,Subsolo
1,I,2.4,0,5,5,100.0,Terreo
2,I,2.4,1,5,5,100.0,1o andar
3,I,5,-1,4,4,100.0,Subsolo
4,I,5,0,3,3,100.0,Terreo
5,I,5,1,4,4,100.0,1o andar
6,M,2.4,-1,6,6,100.0,Subsolo
7,M,2.4,0,6,5,83.3,Terreo
8,M,2.4,1,5,2,40.0,1o andar
9,M,5,-1,4,4,100.0,Subsolo



HIPOTESE (2,4 GHz): descasamento alto nos 2 predios (I = 93%, M = 76%) — compativel com ausencia de gerenciamento adaptativo de canal em nivel de campus, nao com problema de uma edificacao

Reuso de canal entre APs do mesmo predio:


,predio,banda,canal,aps_distintos,grupos_bssid,reuso
0,I,2.4,1,2,"3c:46:a1:66:33:40, c8:a6:08:43:20:a0",True
1,I,2.4,6,5,"3c:46:a1:66:33:30, 3c:46:a1:66:33:40, 70:47:77:74:84:10, 70:47:77:...",True
2,I,2.4,11,1,00:e6:3a:5e:4e:a0,False
3,I,5,40,1,00:e6:3a:9e:4e:a0,False
4,I,5,52,1,c8:a6:08:83:20:a0,False
5,I,5,60,1,70:47:77:b4:55:60,False
6,I,5,108,1,3c:46:a1:a6:33:40,False
7,I,5,124,1,00:e6:3a:8a:81:70,False
8,M,2.4,1,3,"84:18:3a:30:e1, e0:10:7f:3d:ea, e0:10:7f:7d:ea",True
9,M,2.4,6,1,e0:10:7f:3c:c6,False



Distribuicao de pct_melhor_canal:


,predio,banda,n,minimo,mediana,media,maximo
0,I,2.4,15,0,33.0,36.0,100
1,I,5,11,100,100.0,100.0,100
2,M,2.4,17,0,15.0,20.4,50
3,M,5,13,32,100.0,94.8,100



SEMANTICA PENDENTE: 'pct_melhor_canal' tem interpretacao ambigua (qualidade do melhor canal vs. ocupacao do melhor canal). O pipeline a trata APENAS como metrica relativa comparativa, nunca como grandeza fisica. Defina explicitamente no relatorio antes de citar qualquer valor absoluto.


---
## 6. Atenuacao por obstaculo e mapas de calor

`L_obstaculo <= 0` significa que um ponto declarado obstruido mediu sinal igual ou
melhor que o previsto. O pipeline **sinaliza**; recategorizar e decisao de quem leu as
anotacoes de campo.

Nos mapas: com poucas posicoes distintas por pavimento sai **apenas scatter**. Nao ha
krigagem, nao ha contorno suave e nao ha extrapolacao — superficie continua a partir
de 4 ou 5 amostras seria ficcao visual.

In [8]:
print("Atenuacao por obstaculo:")
display(S["obstaculos"])

print("\nMapas gerados:")
display(S["heatmaps"])

Atenuacao por obstaculo:


,predio,banda,ponto_id,pavimento,local,obstaculos,dist_m,rssi_previsto_dbm,rssi_medido_dbm,L_obstaculo_db,incoerente,cenario_alpha
0,I,2.4,I-44,-1,Canto da Biblioteca,Prateleiras de metal/Colunas,6.0,-55.2,-56.0,0.8,False,A
1,I,2.4,I-35,1,Escada,Paredes,9.0,-58.6,-59.0,0.4,False,A
2,I,2.4,I-40,0,Entrada,Portoes de metal,9.0,-58.6,-59.0,0.4,False,A
3,I,2.4,I-45,-1,Escada,Paredes,8.0,-57.6,-58.0,0.4,False,A
4,I,2.4,I-39,0,Escada,Paredes,6.0,-55.2,-55.0,-0.2,True,A
5,M,2.4,M-05,1,Escada de incendio,"Parede, porta de incendio",6.0,-53.2,-70.0,16.8,False,A
6,M,2.4,M-16,-1,Escada,Parede/porta metalica,8.0,-57.3,-68.0,10.7,False,A
7,M,2.4,M-10,0,Escada,Parede/porta metalica,6.0,-53.2,-58.0,4.8,False,A
8,M,2.4,M-01,1,Fundo da M102,"Parede, porta",8.5,-58.1,-60.0,1.9,False,A
9,M,2.4,M-06,0,Sala entre AP1 e AP2,Parede/porta,4.0,-47.4,-46.0,-1.4,True,A



Mapas gerados:


,predio,pavimento,banda,figura,n,situacao,motivo
0,M,-1,2.4,fig_rssi_dbm_M_2.4GHz_pav-1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
1,M,-1,5,fig_rssi_dbm_M_5GHz_pav-1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
2,M,0,2.4,fig_rssi_dbm_M_2.4GHz_pav0.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
3,M,0,5,fig_rssi_dbm_M_5GHz_pav0.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
4,M,1,2.4,fig_rssi_dbm_M_2.4GHz_pav1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
5,M,1,5,fig_rssi_dbm_M_5GHz_pav1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
6,I,-1,2.4,fig_rssi_dbm_I_2.4GHz_pav-1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
7,I,-1,5,fig_rssi_dbm_I_5GHz_pav-1.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
8,I,0,2.4,fig_rssi_dbm_I_2.4GHz_pav0.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."
9,I,0,5,fig_rssi_dbm_I_5GHz_pav0.png,2,gerada,"APENAS SCATTER: 2 posicao(oes) distinta(s), abaixo do minimo 4 par..."


---
## 7. Limitacoes

Montadas a partir dos dados reais da execucao. Arquivo completo em
`saidas/limitacoes.md`.

In [9]:
Markdown(S["limitacoes"])

# Limitacoes do estudo

Documento gerado automaticamente em 18/08/2026 17:43 a partir de dados/leituras.csv. Os numeros abaixo vem da execucao, nao de texto fixo.


## 1. Amostra unica por ponto, sem repeticao nem media

As 56 leituras correspondem a 56 combinacoes distintas de (predio, pavimento, banda, local), com **uma medicao por combinacao**. Nao ha repeticao temporal nem media, entao nao ha como separar variacao de curto prazo (fast fading, ocupacao do meio) do efeito que se quer medir. Todo intervalo de confianca reportado descreve a dispersao ENTRE pontos, nunca a repetibilidade de um ponto.


## 2. Altura de medicao nao controlada

A coluna altura_medicao_m esta vazia em **56 das 56 leituras** (100%). A campanha registrou o efeito uma unica vez, no ponto **M-02**:


> Anotado em campo como '< 1 m'. Em pe os valores chegavam proximos de -30 dBm; no chao, proximos de -45 dBm. Altura da medicao nao registrada.


Sao **15 dB de variacao produzidos por uma variavel que nao foi registrada**. Para comparacao: a perda atribuida a porta corta-fogo neste mesmo estudo e da ordem de 21 dB, e a diferenca entre os cenarios de alpha avaliados vale poucos dB ao longo de toda a faixa de distancias. Ou seja, **a variavel nao controlada excede varios dos efeitos que o estudo tenta medir**.


## 3. BSSID identifica o AP dominante, nao o AP associado

Os BSSIDs foram lidos da lista de varredura do aplicativo (aba de pontos de acesso), que mostra o **AP visivel dominante** no ponto. Nao ha registro de a qual AP o cliente estava efetivamente associado.


Consequencia direta: a expressao *sticky client* **nao e sustentavel por este dado**. O que se pode afirmar e que o AP mais forte na varredura era o indicado, nao que o aparelho estivesse preso a ele.


Cobertura de BSSID por predio:


| predio   |   leituras |   com_bssid |   pct |
|:---------|-----------:|------------:|------:|
| I        |         26 |          26 | 100   |
| M        |         30 |           5 |  16.7 |


## 4. O SINR calculado e um limite pessimista

O piso de ruido usado e **-95 dBm, adotado e nao medido**, uniforme para todos os pontos. Alem disso, tratar interferencia co-canal como ruido aditivo e conservador demais para Wi-Fi: em CSMA/CA a interferencia co-canal atua principalmente por **disputa de airtime** (o transmissor espera o meio ficar livre), e nao somando potencia ao denominador. A capacidade de Shannon derivada dai e teto teorico, jamais previsao de vazao.


## 5. Alpha reportado com 2 algarismos significativos

Pelos motivos das secoes 1 e 2: amostra unica, distancia anotada com aproximacao na origem, altura nao controlada e RSSI 802.11 variando tipicamente +-5 a 10 dB por fast fading. Reportar alpha = 2,62 sugere precisao de centesimos que a amostra nao sustenta; o pipeline reporta alpha ~= 2,6.


Amplitude dos intervalos de confianca obtidos nesta execucao:


| predio   |   banda | cenario   |   alpha |   ic95_inf |   ic95_sup |   n |   amplitude_ic |
|:---------|--------:|:----------|--------:|-----------:|-----------:|----:|---------------:|
| I        |     2.4 | A         |     1.9 |       1.07 |       2.81 |   8 |           1.74 |
| M        |     2.4 | A         |     3.3 |       1.91 |       4.62 |   9 |           2.71 |


Um IC de amplitude comparavel ao proprio valor de alpha confirma que o segundo algarismo ja e o limite do que a amostra sustenta.


## 6. Cobertura desigual entre os predios

| predio   |   leituras |   com_bssid |   xy_declarada |   dist_zero | mapa_ap   | fabricante_ap   |
|:---------|-----------:|------------:|---------------:|------------:|:----------|:----------------|
| I        |         26 |          26 |              0 |           4 | nao       | a confirmar     |
| M        |         30 |           5 |              0 |           0 | sim       | Ruckus Wireless |


A comparacao entre predios herda essas assimetrias. Onde um predio tem mapeamento de AP e o outro nao, a mesma analise nao roda dos dois lados, e a diferenca observada pode ser de **cobertura de dado**, nao de propagacao.


## 7. Equipamento nao totalmente identificado

- **Predio I**: fabricante **nao confirmado**. Os OUIs observados nao foram verificados contra a base do IEEE, e o pipeline nao os infere.

- **Predio M**: fabricante declarado (Ruckus Wireless), modelo nao registrado. Potencia de transmissao e ganho de antena variam entre modelos do mesmo fabricante, entao a ressalva de comparabilidade permanece aberta.


## 8. Transcricao de BSSID nao verificada na fonte

1 par(es) de BSSID diferem em poucos digitos sem cair no mesmo AP fisico. A hipotese de erro de transcricao **nao foi confirmada contra os prints originais do aplicativo**: ela e apenas a leitura mais provavel do padrao observado.


- predio M: e0:10:7f:3d:ea:78 (leitura [7]) vs e0:10:7f:7d:ea:79 (leitura [14]); candidato a erro: e0:10:7f:7d:ea:79


---
## 8. Verificacao contra as referencias historicas

Se estes numeros divergirem, o problema e de parsing ou de mapeamento de colunas —
**nao e descoberta**.

In [10]:
display(S["verificacao"])
print(f"L_obstaculo porta corta-fogo : {S['L_obstaculo_corta_fogo']} dB "
      f"(esperado {PARAMS['referencias']['L_obstaculo_porta_corta_fogo_db']}) "
      f"-> {'OK' if S['L_obstaculo_ok'] else 'DIVERGE'}")
print()
print("TODAS AS VERIFICACOES PASSARAM" if S["verificacao_ok"]
      else "ATENCAO: ha divergencia — investigar antes de usar os resultados")

,verificacao,esperado,obtido,tolerancia,ok
0,alpha M 2.4 GHz (cenario historico),2.62,2.600,0.10,True
1,R2 M 2.4 GHz (cenario historico),0.91,0.906,0.03,True
2,n M 2.4 GHz (cenario historico),9.00,9.000,0.00,True
3,mismatch de canal M 2.4 GHz (%),76.00,76.500,1.00,True


L_obstaculo porta corta-fogo : 21.1 dB (esperado 21.1) -> OK

TODAS AS VERIFICACOES PASSARAM
